# 04 · Pretrained small VLMs: SmolVLM-256M + LoRA on the robot task

**Run this on your NVIDIA GPU machine.** It downloads ~0.5 GB of weights and fine-tunes for a few minutes.

Our TinyVLM learned the task but only knows ~60 words. A **pretrained** VLM already knows language and generic vision (colors, shapes, rooms). We only need to teach it our *skill format*. This is how real robot VLMs are built: start from a foundation model and fine-tune it on robot data.

### Which small VLM?
| Model | Params | FP16 weights | Fits Orin Nano 8GB? | Notes |
|---|---|---|---|---|
| **SmolVLM-256M-Instruct** | 0.26B | ~0.5 GB | easily | used here; fast to fine-tune |
| SmolVLM-500M-Instruct | 0.5B | ~1 GB | easily | better language |
| SmolVLM2-2.2B-Instruct | 2.2B | ~4.4 GB | INT4/INT8 | video support |
| Qwen2.5-VL-3B / newer Qwen-VL | 3-4B | ~7.5 GB | INT4 only | strong grounding; supported by TensorRT Edge-LLM |
| Cosmos Reason (NVIDIA) | 2B+ | varies | small variant, quantized | physical-AI reasoning VLM, made for robots |

Check each model card for the current versions and licenses. To train a small VLM *from scratch* at real scale, see Hugging Face [nanoVLM](https://github.com/huggingface/nanoVLM). It is the grown-up version of notebook 01.

In [ ]:
# !pip install -U 'transformers>=4.46' accelerate peft num2words
import sys, pathlib
ROOT = pathlib.Path.cwd().resolve()
if ROOT.name == 'notebooks':
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / 'src'))

import time, torch
from PIL import Image
import transformers
from transformers import AutoProcessor
try:
    from transformers import AutoModelForImageTextToText as AutoVLM
except ImportError:  # older transformers
    from transformers import AutoModelForVision2Seq as AutoVLM

from edge_vlm.actions import parse_action
from edge_vlm.agent import evaluate_policy
from edge_vlm.data import generate_samples

assert torch.cuda.is_available(), 'This notebook is meant for an NVIDIA GPU machine'
device = torch.device('cuda')
DTYPE = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16
MODEL_ID = 'HuggingFaceTB/SmolVLM-256M-Instruct'
print('transformers', transformers.__version__, '| GPU', torch.cuda.get_device_name(0), '| dtype', DTYPE)

In [ ]:
processor = AutoProcessor.from_pretrained(MODEL_ID)
# One 512px tile per image instead of splitting into many crops:
# 64 visual tokens instead of ~1000. Our 64px camera does not need more.
processor.image_processor.do_image_splitting = False
model = AutoVLM.from_pretrained(MODEL_ID, torch_dtype=DTYPE).to(device).eval()
n = sum(p.numel() for p in model.parameters())
print(f'{n/1e6:.0f}M params, {n * torch.finfo(DTYPE).bits / 8 / 1e9:.2f} GB in {DTYPE}')
print('memory allocated:', round(torch.cuda.memory_allocated() / 1e9, 2), 'GB')

## Prompting: describe the skills, ask for exactly one

The same `build_prompt` text the tiny model sees, plus an instruction that lists the skills. Large models are often asked for JSON. Our `parse_action` accepts both JSON and the compact form.

In [ ]:
INSTRUCTION = ('You are the planner of a home robot. Skills: navigate <room> (kitchen, bedroom, office, living_room), '
               'pick <color> <object>, return, handover. Reply with exactly one skill.')

def to_pil(image):
    return Image.fromarray(image).resize((384, 384), Image.NEAREST)

def messages(prompt, action=None):
    msgs = [{'role': 'user', 'content': [{'type': 'image'}, {'type': 'text', 'text': INSTRUCTION + '\n' + prompt}]}]
    if action is not None:
        msgs.append({'role': 'assistant', 'content': [{'type': 'text', 'text': action}]})
    return msgs

class HFPolicy:
    def __init__(self, model, processor, max_new_tokens=12):
        self.model, self.processor, self.max_new_tokens = model, processor, max_new_tokens

    @torch.no_grad()
    def predict(self, image, prompt):
        text = self.processor.apply_chat_template(messages(prompt), add_generation_prompt=True)
        inputs = self.processor(text=[text], images=[[to_pil(image)]], return_tensors='pt').to(device)
        inputs['pixel_values'] = inputs['pixel_values'].to(DTYPE)
        out = self.model.generate(**inputs, max_new_tokens=self.max_new_tokens, do_sample=False)
        return self.processor.batch_decode(out[:, inputs['input_ids'].shape[1]:], skip_special_tokens=True)[0].strip()

    def act(self, state):
        return self.predict(state.image, state.prompt)

policy = HFPolicy(model, processor)
val_samples = generate_samples(60, seed=10_000)
for s in val_samples[:5]:
    out = policy.predict(s.image, s.prompt)
    print(f'{s.prompt}\n   expert: {s.action:18s} zero-shot: {out!r}  parsed: {parse_action(out)}')

Zero-shot, a 256M model usually understands the gist but gets the format and the reasoning wrong. Bigger models (2-8B) do much better zero-shot, but they cost 10-30x the memory. **Fine-tuning a small model on our task usually beats prompting a large one, and it still fits on the Orin Nano.**

## LoRA fine-tuning

**LoRA** freezes the base weights and learns a low-rank update $W + BA$ (rank $r \ll d$) for chosen layers. We adapt only the **language model's** linear layers. The vision encoder stays frozen because colors and shapes are already generic. The loss is computed only on the answer tokens.

In [ ]:
from peft import LoraConfig, get_peft_model

lora_cfg = LoraConfig(
    r=16, lora_alpha=32, lora_dropout=0.05,
    target_modules=r'.*text_model.*\.(q_proj|k_proj|v_proj|o_proj|gate_proj|up_proj|down_proj)',
)
model = get_peft_model(model, lora_cfg)
model.print_trainable_parameters()

processor.tokenizer.padding_side = 'right'   # so prompt masking by position is valid

def collate(batch):
    images = [[to_pil(s.image)] for s in batch]
    full = [processor.apply_chat_template(messages(s.prompt, s.action)) for s in batch]
    enc = processor(text=full, images=images, return_tensors='pt', padding=True)
    labels = enc['input_ids'].clone()
    labels[enc['attention_mask'] == 0] = -100
    for i, s in enumerate(batch):
        prompt_only = processor.apply_chat_template(messages(s.prompt), add_generation_prompt=True)
        n = processor(text=[prompt_only], images=[images[i]], return_tensors='pt')['input_ids'].shape[1]
        labels[i, :n] = -100   # no loss on image + instruction tokens
    enc['labels'] = labels
    return enc

In [ ]:
from torch.utils.data import DataLoader

TRAIN_EPISODES, EPOCHS, BATCH, LR = 600, 1, 8, 2e-4
train_samples = generate_samples(TRAIN_EPISODES, seed=0)
dl = DataLoader(train_samples, batch_size=BATCH, shuffle=True, collate_fn=collate)
opt = torch.optim.AdamW([p for p in model.parameters() if p.requires_grad], lr=LR)

model.train()
t, step = time.time(), 0
for ep in range(EPOCHS):
    for batch in dl:
        batch = {k: v.to(device) for k, v in batch.items()}
        batch['pixel_values'] = batch['pixel_values'].to(DTYPE)
        loss = model(**batch).loss
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        opt.step(); opt.zero_grad(set_to_none=True)
        if step % 25 == 0:
            print(f'step {step:4d}/{len(dl) * EPOCHS}  loss {loss.item():.4f}  ({time.time() - t:.0f}s)')
        step += 1
model.eval()
print('peak GPU memory:', round(torch.cuda.max_memory_allocated() / 1e9, 2), 'GB')

In [ ]:
policy = HFPolicy(model, processor)   # the LoRA-wrapped model
hits = sum(policy.predict(s.image, s.prompt) == s.action for s in val_samples)
print(f'open-loop exact match: {hits}/{len(val_samples)}')
print('closed loop:', evaluate_policy(policy, n_episodes=50, seed=777))

# Free-form language: this is what the tiny model cannot do
for cmd in ['could you grab my yellow bottle, I think it is in the bedroom',
            'I am thirsty, bring the blue cup please']:
    s = val_samples[0]
    prompt = f'cmd: {cmd} | at: living_room | holding: nothing | visited: none'
    print(cmd, '->', policy.predict(s.image, prompt))

## Save for deployment

Merge LoRA into the base weights so the deployment runtimes see a normal model. From there:

* **llama.cpp (GGUF):** `convert_hf_to_gguf.py` the LM and `--mmproj` the vision part, quantize to Q8_0/Q4_K_M, run on the Jetson (notebook 06).
* **TensorRT Edge-LLM:** export and quantize on the desktop, build the engine on the Jetson. Check its supported-model list first.
* **PyTorch on Jetson:** simplest path, slowest. Good as a baseline.

In [ ]:
OUT = ROOT / 'checkpoints' / 'smolvlm256m_robot'
merged = model.merge_and_unload()
merged.save_pretrained(OUT)
processor.save_pretrained(OUT)
print('saved merged model to', OUT)

### Exercises
1. Swap `MODEL_ID` for `HuggingFaceTB/SmolVLM-500M-Instruct`. Compare accuracy, memory and latency.
2. Ask for JSON output instead (change the targets to `Action.to_json()`). Is it more or less robust?
3. Increase `TRAIN_EPISODES`. Where does closed-loop success saturate?
4. Try LoRA `r=4` vs `r=64`. How many trainable parameters, and how much accuracy difference?